# Investigação dos dados e consultas SQL

Este notebook lê o arquivo com os dados integrados e realiza consultas SQL para responder perguntas sobre o conjunto de dados para testar as hipóteses definidas na etapa de 'Preparação e integração dos dados'.

**Quantas interrupções ocorreram nos últimos 5 anos? Quais estados da Região Sul tiveram uma proporção maior de interrupções?**

A primeira consulta visa medir o volume de interrupções no período de análise, o que indica se há eventos suficientes para treinar um modelo. Já a segunda compara os estados pela proporção das perturbações em que estiveram envolvidos, para verificar a concentração da parcela das interrupções no Sul do país. Como uma perturbação pode atingir vários estados, essas proporções não somam 100%, e elas mostram onde há mais interrupções, e não onde o risco é maior em relação ao tamanho da rede de cada estado.

In [5]:
from pathlib import Path

import duckdb

interruption_file = Path("../data/raw/ONS/INTERRUPCAO_CARGA.csv")
processed_file = Path("../data/processed/ons_inmet_sul.parquet")

INICIO = "2021-01-01"
FIM = "2025-12-31"

In [6]:
periodo = f"""
    SELECT cod_perturbacao AS perturbacao, id_estado AS estado
    FROM read_parquet('{processed_file}')
    WHERE TRY_CAST(din_interrupcaocarga AS TIMESTAMP) >= CAST('{INICIO}' AS DATE)
      AND TRY_CAST(din_interrupcaocarga AS TIMESTAMP) <  CAST('{FIM}' AS DATE) + INTERVAL 1 DAY
"""

duckdb.sql(f"""
    SELECT SUM(n_perturbacoes) AS perturbacoes
    FROM read_parquet('{processed_file}')
    WHERE data BETWEEN '{INICIO}' AND '{FIM}'
""").df()

,perturbacoes
0,305.0


In [7]:
periodo = f"""
    SELECT cod_perturbacao AS perturbacao, id_estado AS estado
    FROM read_csv_auto('{interruption_file}', delim=';', header=true)
    WHERE id_subsistema = 'S'
      AND TRY_CAST(din_interrupcaocarga AS TIMESTAMP) >= CAST('{INICIO}' AS DATE)
      AND TRY_CAST(din_interrupcaocarga AS TIMESTAMP) <  CAST('{FIM}' AS DATE) + INTERVAL 1 DAY
"""

duckdb.sql(f"""
    WITH periodo AS ({periodo})
    SELECT estado,
           COUNT(DISTINCT perturbacao) AS perturbacoes,
           ROUND(100.0 * COUNT(DISTINCT perturbacao) / (SELECT COUNT(DISTINCT perturbacao) FROM periodo), 1) AS pct_perturbacoes
    FROM periodo
    GROUP BY estado
    ORDER BY perturbacoes DESC
""").df()

,estado,perturbacoes,pct_perturbacoes
0,RS,210,68.9
1,PR,85,27.9
2,SC,19,6.2


Foram retornados 305 perturbações no Subsistema Sul entre 2021 e 2025, cerca de 61 por ano. Como são 1.826 dias no período, no máximo 17% deles têm alguma perturbação (menos, se houver dias com mais de uma). A maioria dos dias não tem interrupção, então a variável-alvo de alta interrupção será desbalanceada. (Mudar para todas as regiões? :/ )

Por estado, o Rio Grande do Sul esteve envolvido em 68,9% das perturbações, o Paraná em 27,9% e Santa Catarina em 6,2%. A soma passa de 100% porque 210, 85 e 19 totalizam 314, mais que as 305 perturbações, e a diferença vem de perturbações que atingiram mais de um estado. A concentração no RS não prova que o estado tenha mais risco, já que a proporção não considera o tamanho da rede de cada estado. Santa Catarina tem poucos eventos comparado aos outros estados (19), o que torna inviável um modelo por estado. (vai ter q prever se vai ter interrupção só na região?)